# Game Theory with Data Science Tables

In this notebook we’ll use python to explore three foundational games in economics and political science:

- **Prisoner’s Dilemma** – conflict between individual and collective incentives  
- **Battle of the Sexes** – coordination when preferences differ  
- **Stag Hunt** – coordination when trust matters

In [ ]:
from datascience import Table
import numpy as np

def show_payoffs(title, movesA, movesB, payoffs):
    """
    Helper function to display a Data 8–style payoff matrix.
    
    payoffs[a_idx][b_idx] = (A_payoff, B_payoff)
    Rows = A's moves, Columns = B's moves
    """
    print(f"\n### {title}")
    
    # Build columns for the table
    # First column is the row labels (A's moves)
    cols = ['Player A / Player B', movesA]
    
    # For each of B's moves, create a column showing payoffs for each A move
    for b_idx, b_move in enumerate(movesB):
        # Extract the column: payoffs for all A moves when B plays b_move
        col_values = [payoffs[a_idx][b_idx] for a_idx in range(len(movesA))]
        cols.extend([b_move, col_values])
    
    return Table().with_columns(*cols)

## How to Read Payoff Matrices

This notebook uses the following convention:

- Rows = Player A's strategies 
- Columns = Player B's strategies 
- Each cell is written as **(A payoff, B payoff)**
  - the **first number** is Player A's payoff 
  - the **second number** is Player B's payoff 

Data layout (how payoffs are stored in code)

- Payoffs are stored as a 2D list: `payoffs[a_idx][b_idx] = (A_payoff, B_payoff)`
  - `a_idx` indexes A's choice (row), `b_idx` indexes B's choice (column)

Annotated example

If Player A has strategies [Cooperate, Defect] and Player B has strategies [Cooperate, Defect], the matrix looks like:

|           | B: Cooperate | B: Defect   |
|-----------|--------------|-------------|
| A: Cooperate | (-1, -1)    | (0, -3)     |
| A: Defect    | (-3, 0)     | (-2, -2)    |

- If A plays Cooperate and B plays Cooperate → look at the cell (row = A: Cooperate, col = B: Cooperate): **(-1, -1)**
  - A gets -1, B gets -1
- If A plays Defect and B plays Cooperate   → cell = **(-3, 0)**
  - A gets -3, B gets 0

Quick checklist when reading a matrix

1. Pick the row (A's move) and the column (B's move).
2. Read the cell at that intersection: first number = A's payoff, second = B's payoff.
3. Feed the same `payoffs` array to the analysis functions (they expect `payoffs[a_idx][b_idx]`).



## 1. Prisoner's Dilemma

### The Story

Two suspects are arrested and interrogated separately. The police don't have enough evidence to convict either on the main charge, but can convict both on a lesser charge. Each prisoner is offered a deal:

- **If you testify against your partner (Defect) and they stay silent (Cooperate)**: You go free, they get 3 years
- **If both stay silent (both Cooperate)**: You each get 1 year on the lesser charge
- **If both testify against each other (both Defect)**: You each get 2 years

### The Dilemma

From each individual's perspective, **defecting is always better** regardless of what the other does:
- If the other cooperates, you get 0 years (defect) vs. 1 year (cooperate)
- If the other defects, you get 2 years (defect) vs. 3 years (defect)

But if both think this way, they both defect and get 2 years each — worse than if they'd both cooperated (1 year each)!

In [ ]:
A_moves = ['Cooperate', 'Defect']
B_moves = ['Cooperate', 'Defect']

# payoffs[a][b] = (A_payoff, B_payoff)
# Rows = A's choice, Columns = B's choice
pd_payoffs = [
    [(-1, -1), (-3, 0)],   # A plays Cooperate: (B=Coop: both get -1), (B=Def: A gets -3, B gets 0)
    [(0, -3), (-2, -2)]    # A plays Defect: (B=Coop: A gets 0, B gets -3), (B=Def: both get -2)
]

pd_matrix = show_payoffs(
    "Prisoner's Dilemma",
    A_moves, B_moves,
    pd_payoffs
)
pd_matrix

### Payoff table (payout to A, payout to B)

| (payout to A, payout to B) / | B: Cooperate | B: Defect |
|---|---:|---:|
| A: Cooperate | (-1, -1) | (-3, 0) |
| A: Defect    | (0, -3)  | (-2, -2) |

Both players face a temptation to defect.  
Even though (Cooperate, Cooperate) gives a better joint outcome,  
each has an incentive to defect — making (Defect, Defect) the only Nash equilibrium.

### Key Insight

This shows the conflict between **individual rationality** and **collective benefit**. It appears everywhere: arms races, environmental policy, public goods, business competition, etc.

## 2. Battle of the Sexes

### The Story

A couple wants to spend the evening together, but they prefer different activities. One prefers Opera, the other prefers Football. They can't communicate before choosing where to go.

- **If both go to the same place**: They're happy to be together (positive payoffs)
- **If they go to different places**: They're both unhappy being alone (zero payoffs)
- **Each prefers their own favorite activity**: Opera-lover gets 2 at Opera vs. 1 at Football (and vice versa)

### The Challenge

Unlike Prisoner's Dilemma, here **coordination is key**. Both players want to meet up, but they disagree on where. There's no dominant strategy — the best choice depends on what the other person does.

In [ ]:
A_moves = ['Opera', 'Football']
B_moves = ['Opera', 'Football']

# payoffs[a][b]
battle_payoffs = [
    [(2, 1), (0, 0)],  # A plays Opera: B=Opera, B=Football
    [(0, 0), (1, 2)]   # A plays Football: B=Opera, B=Football
]

battle_matrix = show_payoffs(
    "Battle of the Sexes",
    A_moves, B_moves,
    battle_payoffs
)
battle_matrix

Here coordination matters.  
Each player wants to meet the other, but prefers a different venue.  
There are **two pure-strategy Nash equilibria**:  
(Opera, Opera) and (Football, Football).

### Key Insight

This represents situations where parties benefit from coordination but have conflicting preferences about which outcome. Examples: technology standards, meeting points, international agreements. The challenge isn't avoiding mutual defection — it's agreeing on which cooperative outcome to choose.

## 3. Stag Hunt

### The Story

Two hunters can either cooperate to hunt a stag (high reward, requires both) or individually hunt a hare (safe, lower reward).

- **If both hunt Stag**: They successfully catch it and each get 4 (best outcome)
- **If one hunts Stag, other hunts Hare**: Stag hunter gets 0 (can't catch it alone), Hare hunter gets 3
- **If both hunt Hare**: Each safely gets 3

### The Trust Problem

Hunting the stag is the best collective outcome, but it's risky. If your partner abandons you for a hare, you get nothing. Hunting hare is safer but less rewarding. The game shows how **lack of trust** can prevent reaching the best outcome.

In [ ]:
A_moves = ['Stag', 'Hare']
B_moves = ['Stag', 'Hare']

# payoffs[a][b] = (A_payoff, B_payoff)
# Rows = A's choice, Columns = B's choice
stag_payoffs = [
    [(4, 4), (0, 3)],  # A plays Stag: (B=Stag: both get 4), (B=Hare: A gets 0, B gets 3)
    [(3, 0), (3, 3)]   # A plays Hare: (B=Stag: A gets 3, B gets 0), (B=Hare: both get 3)
]

stag_matrix = show_payoffs(
    "Stag Hunt",
    A_moves, B_moves,
    stag_payoffs
)
stag_matrix

The **Stag Hunt** shows how trust and risk affect cooperation.  
Two equilibria exist:
- (Stag, Stag): efficient but risky — requires mutual trust  
- (Hare, Hare): safe but less rewarding  

### Key Insight

Unlike Prisoner's Dilemma where individual incentives prevent cooperation, here cooperation IS individually rational — **but only if you trust others to cooperate too**. This represents situations like climate change, R&D investment, or any scenario where the best outcome requires risky mutual commitment. Policy or institutions can help players coordinate on the better equilibrium.

### Teaching Extensions

- Add a function to compute best responses for each player (as earlier).  
- Use `interact()` sliders so students can change payoffs and observe equilibria.  
- Ask students to classify each game as **coordination** or **dilemma** and explain why.

## Computing Best Responses

A **best response** is a player's optimal strategy given what they believe the other player will do. 

To find Nash equilibria, we look for strategy pairs where each player's choice is a best response to the other's choice.

The function below takes a payoff matrix and computes:
- Player A's best response to each of Player B's possible moves
- Player B's best response to each of Player A's possible moves

In [ ]:
def find_best_responses(movesA, movesB, payoffs):
    """
    Find best responses for both players in a 2x2 game where payoffs[a][b] = (A_payoff, B_payoff).
    Rows correspond to A's moves, columns correspond to B's moves.
    """
    print("Player A's Best Responses:")
    for b_idx, b_move in enumerate(movesB):
        print(f"  If B plays {b_move}:")
        # For each A move (row), read the A payoff when B plays b_move (column b_idx)
        for a_idx, a_move in enumerate(movesA):
            a_payoff = payoffs[a_idx][b_idx][0]
            print(f"    {a_move} gives A: {a_payoff}")
        # Find which A move (row) gives the highest A payoff for this B column
        a_payoffs = [payoffs[a_idx][b_idx][0] for a_idx in range(len(movesA))]
        best_a_idx = int(np.argmax(a_payoffs))
        print(f"    → A should play {movesA[best_a_idx]}")
    
    print("\nPlayer B's Best Responses:")
    for a_idx, a_move in enumerate(movesA):
        print(f"  If A plays {a_move}:")
        # For each B move (column), read the B payoff when A plays a_move (row a_idx)
        for b_idx, b_move in enumerate(movesB):
            b_payoff = payoffs[a_idx][b_idx][1]
            print(f"    {b_move} gives B: {b_payoff}")
        # Find which B move (column) gives the highest B payoff for this A row
        b_payoffs = [payoffs[a_idx][b_idx][1] for b_idx in range(len(movesB))]
        best_b_idx = int(np.argmax(b_payoffs))
        print(f"    → B should play {movesB[best_b_idx]}")

### Example: Finding Best Responses in Prisoner's Dilemma

In [ ]:
# Reuse the Prisoner's Dilemma payoffs from above
find_best_responses(
    ['Cooperate', 'Defect'],
    ['Cooperate', 'Defect'], 
    pd_payoffs
)


In [ ]:
# Battle of the Sexes - reuse payoffs from above
find_best_responses(
    ['Opera', 'Football'],
    ['Opera', 'Football'], 
    battle_payoffs
)

In [ ]:
# Stag Hunt - reuse payoffs from above
find_best_responses(
    ['Stag', 'Hare'],
    ['Stag', 'Hare'], 
    stag_payoffs
)

## 4. Your Own Game

Now it's your turn! Create your own 2x2 game by filling in the template below.

Think of a scenario where two players make choices. Some ideas:
- Should two students study together or separately before an exam?
- Should two companies compete on price or quality?
- Should two countries invest in green energy or stick with fossil fuels?
- Should two friends share resources or keep them?

Fill in:
1. The names of the two strategies for each player
2. The payoffs for each combination of choices
3. Run the cells to see your game's payoff matrix and best responses!

In [ ]:
# Step 1: Name your strategies
# Replace these with your own strategy names
my_A_moves = ['Strategy A1', 'Strategy A2']
my_B_moves = ['Strategy B1', 'Strategy B2']

# Step 2: Fill in the payoffs
# Remember: payoffs[a][b] = (A_payoff, B_payoff)
# Rows = A's choice, Columns = B's choice
my_payoffs = [
    [(0, 0), (0, 0)],  # A plays Strategy A1: what happens when B plays B1? B2?
    [(0, 0), (0, 0)]   # A plays Strategy A2: what happens when B plays B1? B2?
]

# Step 3: Display your game
my_game_matrix = show_payoffs(
    "My Game",
    my_A_moves, my_B_moves,
    my_payoffs
)
my_game_matrix

In [ ]:
# Step 4: Find the best responses in your game
find_best_responses(my_A_moves, my_B_moves, my_payoffs)

### Questions to Answer About Your Game

After creating your game, think about:

1. **Does either player have a dominant strategy?** (A strategy that's always best, no matter what the other player does)

2. **What are the Nash equilibria?** (Strategy pairs where each player's choice is a best response to the other's)

3. **Is your game more like:**
   - **Prisoner's Dilemma** (individual incentives conflict with collective good)?
   - **Battle of the Sexes** (both want to coordinate, but prefer different outcomes)?
   - **Stag Hunt** (cooperation is best, but risky without trust)?
   - **Something else?**

4. **What real-world situation does your game represent?**